# GeoPulse — Day 4
## PySpark Data Processing

### Objective

Introduce Apache Spark through PySpark and process the feature-engineered
GeoPulse mobility dataset using Spark DataFrames and transformations.

### Tasks

- Initialize a Spark session
- Load the feature-engineered mobility dataset
- Inspect the Spark DataFrame
- Perform filtering and column operations
- Perform aggregations
- Compare Spark output with the existing Pandas analysis

In [4]:
%pip install pyspark

Note: you may need to restart the kernel to use updated packages.


In [1]:
import subprocess

result = subprocess.run(
    ["java", "-version"],
    capture_output=True,
    text=True
)

print(result.stderr)

java version "17.0.20" 2026-07-21 LTS
Java(TM) SE Runtime Environment (build 17.0.20+7-LTS-191)
Java HotSpot(TM) 64-Bit Server VM (build 17.0.20+7-LTS-191, mixed mode, sharing)



In [8]:
import pyspark

print("PySpark version:", pyspark.__version__)

#Spark Session
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("GeoPulse")
    .master("local[*]")
    .getOrCreate()
)

print("Spark Session created successfully.")

#CSV ko Spark DataFrame mein load karo
spark_df = spark.read.csv(
    "mobility_pings_featured.csv",
    header=True,
    inferSchema=True
)

print("Dataset loaded into Spark successfully.")

#Spark DataFrame inspect karo
spark_df.printSchema()

print("Rows:", spark_df.count())
print("Columns:", len(spark_df.columns))

spark_df.show(5, truncate=False)

#Filter: Peak Hour Mobility
peak_hour_df = spark_df.filter(
    spark_df["is_peak_hour"] == 1
)

print("Peak hour records:", peak_hour_df.count())

peak_hour_df.show(5, truncate=False)

#Select specific columns
selected_df = spark_df.select(
    "ping_id",
    "anonymous_user_id",
    "timestamp",
    "movement_type",
    "time_period",
    "is_peak_hour",
    "is_weekend"
)

selected_df.show(5, truncate=False)

#Movement Type-wise Count
from pyspark.sql import functions as F

movement_summary = (
    spark_df
    .groupBy("movement_type")
    .agg(
        F.count("*").alias("total_pings")
    )
    .orderBy(F.desc("total_pings"))
)

movement_summary.show()

#Time Period-wise Mobility
time_summary = (
    spark_df
    .groupBy("time_period")
    .agg(
        F.count("*").alias("total_pings")
    )
    .orderBy("total_pings", ascending=False)
)

time_summary.show()

#Hour-wise Mobility Aggregation
hourly_summary = (
    spark_df
    .groupBy("hour")
    .agg(
        F.count("*").alias("total_pings")
    )
    .orderBy("hour")
)

hourly_summary.show(24)

#Weekend vs Weekday
weekend_summary = (
    spark_df
    .groupBy("is_weekend")
    .agg(
        F.count("*").alias("total_pings")
    )
    .orderBy("is_weekend")
)

weekend_summary.show()

#Movement Type × Time Period
movement_time_summary = (
    spark_df
    .groupBy("time_period", "movement_type")
    .agg(
        F.count("*").alias("total_pings")
    )
    .orderBy("time_period", F.desc("total_pings"))
)

movement_time_summary.show(20)

#Spark Result Validation
spark_total = spark_df.count()

walking_count = (
    spark_df
    .filter(spark_df["movement_type"] == "Walking")
    .count()
)

print("Total Spark records:", spark_total)
print("Walking records:", walking_count)

#Spark DataFrame Summary
print("===== GeoPulse Day 4 PySpark Summary =====")
print("Total records:", spark_df.count())
print("Total columns:", len(spark_df.columns))
print("Peak-hour records:", peak_hour_df.count())
print("Walking records:", walking_count)

#Stop Spark Session
spark.stop()

print("Spark Session stopped successfully.")

PySpark version: 4.2.0
Spark Session created successfully.
Dataset loaded into Spark successfully.
root
 |-- ping_id: string (nullable = true)
 |-- anonymous_user_id: string (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- latitude: double (nullable = true)
 |-- longitude: double (nullable = true)
 |-- device_type: string (nullable = true)
 |-- movement_type: string (nullable = true)
 |-- date: date (nullable = true)
 |-- hour: integer (nullable = true)
 |-- day_of_week: string (nullable = true)
 |-- time_period: string (nullable = true)
 |-- is_peak_hour: integer (nullable = true)
 |-- is_weekend: integer (nullable = true)
 |-- month: integer (nullable = true)
 |-- week_of_month: integer (nullable = true)
 |-- movement_type_code: integer (nullable = true)

Rows: 5000
Columns: 16
+-------+-----------------+-------------------+---------+---------+-----------+-------------+----------+----+-----------+-----------+------------+----------+-----+-------------+--------------

# Day 4 Conclusion

PySpark was successfully introduced into the GeoPulse data-processing
workflow.

### Tasks Completed

- Initialized a local Spark session
- Loaded the feature-engineered mobility dataset into a Spark DataFrame
- Inspected Spark schema and dataset structure
- Performed filtering operations
- Selected relevant columns
- Aggregated mobility records by movement type
- Aggregated mobility records by time period
- Analyzed hourly mobility activity
- Compared weekday and weekend mobility
- Analyzed movement type across different time periods
- Validated Spark results against the existing Pandas analysis

### Validation

The Spark DataFrame contained 5,000 mobility records.

The movement-type aggregation produced the same overall counts observed
during the earlier Pandas-based analysis, confirming consistency between
the two processing approaches.

### Day 4 Status

Completed.

PySpark processing establishes the distributed data-processing foundation
for the upcoming spatial analytics stage using Apache Sedona.

All mobility data used in this project is synthetic and created for
educational purposes.